# gnome-ai-assistant: embed your files on a GPU

Run this on a free **Colab GPU** (Runtime > Change runtime type > GPU) when indexing thousands of files would take hours on your laptop's CPU.

1. On your computer run `aiassistant index --export` (creates `~/.cache/gnome-ai-assistant/bundle.zip`).
2. Run all cells here and upload that zip when asked.
3. Download `vectors.npy`, then run `aiassistant index --import vectors.npy`.

**Privacy:** the bundle contains your document text and downscaled images/video frames; they are uploaded to Google's Colab for the duration of the run. Delete the Colab runtime afterwards (Runtime > Disconnect and delete runtime).

In [ ]:
!pip -q install -U sentence-transformers transformers torchvision pillow

In [ ]:
from google.colab import files
up = files.upload()  # pick bundle.zip
!unzip -q -o bundle.zip -d bundle

In [ ]:
import json, torch, numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
meta = json.load(open('bundle/items.json'))
items = meta['items']; model_id = meta['model']
# never float16 (EmbeddingGemma gives NaNs); T4 has no native bf16, so float32 there
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float32
kw = {'model_kwargs': {'torch_dtype': dtype}}
if model_id == 'google/embeddinggemma-2':
    kw['config_kwargs'] = {'audio_config': None}   # text + image + video frames, no audio encoder
model = SentenceTransformer(model_id, device='cuda', **kw)
print(len(items), 'items with', model_id, dtype)

In [ ]:
from tqdm.auto import tqdm
dim = model.get_sentence_embedding_dimension()
out = np.zeros((len(items), dim), dtype=np.float32)
text_idx = [i for i, it in enumerate(items) if it['kind'] == 'text']
media_idx = [i for i, it in enumerate(items) if it['kind'] != 'text']
B = 64
for s in tqdm(range(0, len(text_idx), B), desc='text'):
    ix = text_idx[s:s+B]
    out[ix] = model.encode([items[i]['text'] for i in ix], normalize_embeddings=True, batch_size=B, show_progress_bar=False).astype(np.float32)
B = 16
for s in tqdm(range(0, len(media_idx), B), desc='images / video frames'):
    ix = media_idx[s:s+B]
    ims = []
    for i in ix:
        with Image.open('bundle/' + items[i]['media']) as im:
            ims.append({'image': im.convert('RGB')})
    out[ix] = model.encode(ims, normalize_embeddings=True, batch_size=B, show_progress_bar=False).astype(np.float32)
assert np.isfinite(out).all(), 'NaN/inf in embeddings'
np.save('vectors.npy', out)
print('done', out.shape)

In [ ]:
files.download('vectors.npy')